# 🎛️ Task 16: Hyperparameter Tuning (Titanic)
**Project:** Optimizing Random Forest for Survival Prediction  
**Author:** [Your Name]

## 🎯 Objective
To demonstrate that **Hyperparameter Tuning** can improve model performance.
We will:
1.  Train a **Default Random Forest** (Baseline).
2.  Use **GridSearchCV** to find the optimal `n_estimators`, `max_depth`, and `min_samples_split`.
3.  Compare the results.

## 📂 Dataset
* **Source:** Titanic Dataset (via Seaborn).
* **Challenge:** The dataset is noisy. Default models often overfit (memorize the training data) and perform poorly on the test data. Tuning helps regularize the model.

In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# 1. Load Dataset
df = sns.load_dataset('titanic')

# 2. Basic Preprocessing
# Drop columns that are redundant or have too many missing values
df.drop(['deck', 'embark_town', 'alive', 'who', 'adult_male', 'class'], axis=1, inplace=True)

# Fill Missing Values
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# Encode Categorical Variables (Sex, Embarked)
le = LabelEncoder()
df['sex'] = le.fit_transform(df['sex'])
df['embarked'] = le.fit_transform(df['embarked'])

# 3. Split Data
X = df.drop('survived', axis=1)
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data Prepared. Train Shape: {X_train.shape}")

Data Prepared. Train Shape: (712, 8)


In [6]:
# 4. Baseline Model (Default Hyperparameters)
# By default, n_estimators=100 and max_depth=None (it grows until pure)
baseline_rf = RandomForestClassifier(random_state=42)
baseline_rf.fit(X_train, y_train)

# Predict & Evaluate
baseline_pred = baseline_rf.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)

print(f"Baseline Accuracy (Default): {baseline_acc:.4f}")
print("(This model might be overfitting because max_depth is unlimited)")

Baseline Accuracy (Default): 0.8268
(This model might be overfitting because max_depth is unlimited)


In [7]:
# 5. Define Parameter Grid
# We are testing 3 x 3 x 2 = 18 combinations
param_grid = {
    'n_estimators': [50, 100, 200],      # Number of trees
    'max_depth': [None, 10, 20],         # How deep each tree can grow
    'min_samples_split': [2, 5, 10],     # Minimum samples to split a node
    'criterion': ['gini', 'entropy']     # How to measure quality of a split
}

# 6. Run GridSearchCV
# cv=5 means 5-Fold Cross Validation (robust testing)
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, verbose=1, n_jobs=-1)

print("Starting Grid Search...")
grid.fit(X_train, y_train)
print("Search Complete!")

Starting Grid Search...
Fitting 5 folds for each of 54 candidates, totalling 270 fits
Search Complete!


In [8]:
# 7. Get Best Results
best_rf = grid.best_estimator_
best_params = grid.best_params_

print(f"\n🏆 Best Parameters: {best_params}")

# 8. Evaluate Tuned Model
tuned_pred = best_rf.predict(X_test)
tuned_acc = accuracy_score(y_test, tuned_pred)

print(f"Tuned Accuracy: {tuned_acc:.4f}")

# 9. Comparison Table
comparison = pd.DataFrame({
    'Model': ['Default Random Forest', 'Tuned Random Forest'],
    'Accuracy': [baseline_acc, tuned_acc],
    'Improvement': [0.0, tuned_acc - baseline_acc]
})

print("\n--- Performance Comparison ---")
print(comparison)


🏆 Best Parameters: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 50}
Tuned Accuracy: 0.8380

--- Performance Comparison ---
                   Model  Accuracy  Improvement
0  Default Random Forest  0.826816     0.000000
1    Tuned Random Forest  0.837989     0.011173


## 🧠 Final Evaluation

### 1. The Improvement
* **Default Model:** Accuracy **82.68%**. By default, Random Forest trees grow indefinitely (`max_depth=None`), which can lead to overfitting by memorizing noise in the training data.
* **Tuned Model:** Accuracy **83.80%**. By limiting tree depth and requiring more samples to split, the Grid Search forced the model to generalize better to unseen passengers.

### 2. Best Parameters Found
The Grid Search identified the optimal settings:
* **Criterion:** `entropy` (Worked better than the default 'gini').
* **Max Depth:** `10` (Prevented the trees from becoming too complex).
* **Min Samples Split:** `5` (Required at least 5 passengers in a group to justify a split, reducing noise).
* **N Estimators:** `50` (Proving that sometimes fewer, better-tuned trees are enough).

### 3. Conclusion
We successfully improved the model performance by **+1.12%**. While this seems small, in competitive machine learning (like Kaggle), bridging the gap from ~82% to ~84% is often the hardest part and separates the top models from the average ones.